## Summary

There are 7032 customer records where 1869 customers cancel their subscription with Telco.

Balanced columns:
- Gender
- Partner
- Payment Method




## Libraries

In [0]:
from pyspark.sql.functions import when, col, count
import seaborn as sns
sns.set_theme()
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import numpy as np
import math
import scipy.stats as ss
from scipy.stats import pointbiserialr

## Load data

In [0]:
# Read bronze layer data
clean_data = spark.table('workspace.telco.bronze_data')
display(clean_data.limit(5)) # Sample

## EDA

### Number of records

In [0]:
# Number of records
clean_data.select(count("customerID")).show()

### Distribution of the Categorical Columns

In [0]:
def plot_cat_distribution(df:pd.DataFrame, column:str):
    """ 
        Plot the distribution of a column in a dataframe
        Input:
            df (pandas dataframe): Data
            column (str): column name
        
    """
    # Show column count
    df.groupBy(column).count().show()
    # Column Count
    column_count = df.groupBy(column).count().toPandas()
    plt.figure(figsize=(10, 5))
    # Pie chart
    plt.pie(data=column_count, x="count", labels=column, autopct="%.2f%%");
    plt.show()

In [0]:
# Categorical columns
categorical_cols_1 = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity"]
categorical_cols_2 = ["OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod", "Churn"]
categorical_cols = categorical_cols_1 + categorical_cols_2
# Numerical columns
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

In [0]:
# Plot categorical columns distribution
for cat in categorical_cols:
    print(20*"*" + f" Column: {cat} " + 20*"*")
    plot_cat_distribution(clean_data, cat)

### Numerical Distribution

In [0]:
# Plot monthly charges distribution
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data.toPandas(), x="Contract", y="MonthlyCharges", hue="Contract")
plt.title("Monthly Charges by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Monthly Charges")
plt.show()

In [0]:
# Plot total charges distribution
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data.toPandas(), x="Contract", y="TotalCharges", hue="Contract")
plt.title("Total Charges by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Total Charges")
plt.show()

In [0]:
# Plot tenure distribution
plt.figure(figsize=(10, 5))
sns.boxplot(data=clean_data.toPandas(), x="Contract", y="tenure", hue="Contract")
plt.title("Tenure by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Tenure")
plt.show()

### Contract outliers in two year contract

In [0]:

from pyspark.sql.functions import col
outliers_clean_data = clean_data.filter(col("tenure") <= 15).filter(col("Contract")=="Two year").filter("Churn" == "1")

## Correlation among features

### Encoding categorical features

In [0]:
# encoder
le = LabelEncoder()
# convert to pandas to avoid Spark OOM (Databricks Free Edition limitation)
clean_data_df = clean_data.toPandas()
# Applying label encoder to each categorical column
for col in categorical_cols:
    clean_data_df[col] = le.fit_transform(clean_data_df[col])
display(clean_data_df.head()) # Results


In [0]:
# encoder
le = LabelEncoder()
# convert to pandas to avoid Spark OOM (Databricks Free Edition limitation)
outliers_clean_data_df = outliers_clean_data.toPandas()
# Applying label encoder to each categorical column
for col in categorical_cols:
    outliers_clean_data_df[col] = le.fit_transform(outliers_clean_data_df[col])
display(outliers_clean_data_df.head()) # Results

In [0]:
# Create Spark Dataframe
clean_data = spark.createDataFrame(clean_data_df)
# Save encoded data as a table
clean_data.write.option("overwrite", True).saveAsTable("workspace.telco.indexed_data") 

In [0]:
# Create Spark Dataframe
outliers_clean_data = spark.createDataFrame(outliers_clean_data_df)

## Correlation Analysis

### Numerical features

In [0]:
# Assemble the features into a single vector column
assembler = VectorAssembler(
    inputCols=["tenure", "MonthlyCharges", "TotalCharges"],
    outputCol="features"
)
# Transform data to a single vector column
vector_data = assembler.transform(clean_data).select("features")

# Compute the Pearson correlation matrix (among numerical features)
corr_matrix = Correlation.corr(vector_data, "features", "pearson").head()

# Correlation DenseMatrix
print("Pearson correlation matrix:\n" + str(corr_matrix[0]))


In [0]:
out_assembler = VectorAssembler(
    inputCols=["tenure", "MonthlyCharges", "TotalCharges"],
    outputCol="features"
)
# Transform data to a single vector column
out_vector_data = out_assembler.transform(outliers_clean_data).select("features")

# Compute the Pearson correlation matrix (among numerical features)
out_corr_matrix = Correlation.corr(out_vector_data, "features", "pearson").head()

# Correlation DenseMatrix
print("Pearson correlation matrix:\n" + str(out_corr_matrix[0]))

In [0]:
# Convert DenseMatrix to a numpy array
corr_array = np.array(corr_matrix[0].toArray())

# Convert to Pandas DataFrame
corr_df = pd.DataFrame(corr_array, columns=numerical_cols, index=numerical_cols)

# Plot correlation heat map
plt.figure(figsize=(10,10))
sns.heatmap(corr_df, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix among Numerical Features")
plt.show()


In [0]:
# Convert DenseMatrix to a numpy array
out_corr_array = np.array(out_corr_matrix[0].toArray())

# Convert to Pandas DataFrame
out_corr_df = pd.DataFrame(out_corr_array, columns=numerical_cols, index=numerical_cols)

# Plot correlation heat map
plt.figure(figsize=(10,10))
sns.heatmap(out_corr_df, annot=True, cmap="coolwarm")
plt.title("Correlation Matrix among Numerical Features")
plt.show()

### Categorical features

In [0]:
# Define Cramér's V function
def cramers_v(df, col1, col2):
    """
        Computes Cramér's V correlation coefficient between two categorical columns.
        Args:
            df: Spark DataFrame containing the two columns.
            col1: First categorical column.
            col2: Second categorical column.
        Returns:
            Cramér's V correlation coefficient between col1 and col2.
    """
    contingency = df.groupBy(col1, col2).count().toPandas().pivot(index=col1, columns=col2, values='count').fillna(0)
    chi2 = ss.chi2_contingency(contingency)[0]       # chi-square
    n = contingency.sum().sum()
    phi2 = chi2 / n
    r, k = contingency.shape
    return math.sqrt(phi2 / min(k - 1, r - 1))       # Cramér's V coefficient


In [0]:
# Applying Cramer's V to each pair of categorical columns
results = {}

for c1 in categorical_cols: # Iterate over categorical columns
    results[c1] = {}
    for c2 in categorical_cols: # Iterate over categorical columns
        if c1 == c2:
            results[c1][c2] = 1.0 # Same column -> correlation = 1
        else:
            results[c1][c2] = cramers_v(clean_data, c1, c2)  # Compute Cramér's V

cat_corr_df = pd.DataFrame(results)
# Plot correlation heat map
plt.figure(figsize=(12,10))
sns.heatmap(cat_corr_df, annot=True, cmap="Blues", fmt=".2f", 
            annot_kws={"size":8}) 
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title("Cramér's V — Categorical Correlation")
plt.show()

In [0]:
# Applying Cramer's V to each pair of categorical columns
results = {}

for c1 in categorical_cols: # Iterate over categorical columns
    results[c1] = {}
    for c2 in categorical_cols: # Iterate over categorical columns
        if c1 == c2:
            results[c1][c2] = 1.0 # Same column -> correlation = 1
        else:
            results[c1][c2] = cramers_v(outliers_clean_data, c1, c2)  # Compute Cramér's V

cat_corr_df = pd.DataFrame(results)
# Plot correlation heat map
plt.figure(figsize=(12,10))
sns.heatmap(cat_corr_df, annot=True, cmap="Blues", fmt=".2f", 
            annot_kws={"size":8}) 
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.title("Cramér's V — Categorical Correlation")
plt.show()

## Categorical-Numerical Features

In [0]:
# Define biserial correlation function
def point_biserial(df, cat, num):
    """
        Computes point-biserial correlation between a categorical column and a numerical column.
        Args:
            df: Spark DataFrame containing the two columns.
            cat: Categorical column with two unique categories.
            num: Numerical column.
        Returns:
            Point-biserial correlation between cat and num.
    """
    # Select the categorical column and numerical column as Pandas DataFrames
    pdf = df.select(cat, num).dropna().toPandas()
    return pointbiserialr(pdf[cat], pdf[num]).correlation # Compute biserial correlation


In [0]:
# Define correlation ratio function
def eta_squared(df, cat, num):
    """
        Calculates eta-squared for a categorical variable (with more than two unique categories) and a numeric variable.
        Args:
            df: Spark DataFrame
            cat: categorical variable with more than two unique categories
            num: numeric variable
        Returns:
            eta-squared value
    
    """
    # Select the categorical column and numerical column as Pandas DataFrames
    pdf = df.select(cat, num).dropna().toPandas()
    # Select uniques values of the categorical column
    categories = pdf[cat].unique()
    # Convert the numerical column values to numpy array
    y = pdf[num].values
    grand_mean = np.mean(y) # Compute the grand mean
    # Compute the sum of squares between groups
    ss_between = sum([
        len(pdf[pdf[cat] == c]) * (np.mean(pdf[pdf[cat] == c][num]) - grand_mean)**2
        for c in categories
    ])
    ss_total = sum((y - grand_mean)**2)

    return ss_between / ss_total if ss_total > 0 else 0


In [0]:
# Applying biserial or correlation ratio correlation to each pair of categorical and numerical columns
results = {}

for cat in categorical_cols: # Iterate over categorical columns
    results[cat] = {}
    unique_count = clean_data.select(cat).distinct().count() # Count unique values
    for num in numerical_cols: # Iterate over numerical columns
        if unique_count == 2:   # binary → point biserial
            results[cat][num] = point_biserial(clean_data, cat, num)
        else:                   # multiclass → correlation ratio
            results[cat][num] = eta_squared(clean_data, cat, num)

mixed_corr_df = pd.DataFrame(results).T

# Plot correlation heat map
plt.figure(figsize=(10,6))
sns.heatmap(mixed_corr_df, annot=True, cmap="coolwarm")
plt.title("Mixed Correlation: Numerical × Categorical Features")
plt.show()